In [862]:
import pandas as pd
import warnings
warnings.simplefilter('ignore')
print('succes')

succes


In [863]:
import pyodbc

DB1  = {
    'servername' : r'LAPTOP-28CE8V5M\SQLEXPRESS',
    'database' : 'DWH Project DB'
}

export_conn1 = pyodbc.connect(
    'DRIVER={SQL SERVER};SERVER=' +
    DB1['servername'] + 
    ';DATABASE=' + 
    DB1['database'] + 
    ';Trusted_Connection=yes'
)

export_cursor_result = export_conn1.cursor()

print('done result database')

DB2  = {
    'servername' : r'LAPTOP-28CE8V5M\SQLEXPRESS',
    'database' : 'SDM Project DB'
}

export_conn_SDM = pyodbc.connect(
    'DRIVER={SQL SERVER};SERVER=' +
    DB2['servername'] + 
    ';DATABASE=' + 
    DB2['database'] + 
    ';Trusted_Connection=yes'
)

export_cursor_SDM = export_conn_SDM.cursor()

print('done SDM Project DB database')

done result database
done SDM Project DB database


In [864]:
export_cursor_result.execute("EXEC sp_MSforeachtable 'ALTER TABLE ? NOCHECK CONSTRAINT ALL'")

export_cursor_result.execute("SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_TYPE = 'BASE TABLE'")
tables = export_cursor_result.fetchall()

for table in tables:
    table_name = table[0]
    export_cursor_result.execute(f"DELETE FROM {table_name}")
    print(f"Emptied table: {table_name}")

export_cursor_result.execute("EXEC sp_MSforeachtable 'ALTER TABLE ? WITH CHECK CHECK CONSTRAINT ALL'")

export_conn1.commit()

Emptied table: Person_Person
Emptied table: Person_Address
Emptied table: Products
Emptied table: Production_BillOfMaterials
Emptied table: Sales_SalesTerritory
Emptied table: Employees
Emptied table: Customers
Emptied table: Orders
Emptied table: Order_Details
Emptied table: CustomerCustomerDemo
Emptied table: Bonus
Emptied table: EmployeeTerritories
Emptied table: Purchasing_PurchaseOrderHeader
Emptied table: Purchasing_PurchaseOrderDetail


In [865]:
person_dimensie = pd.read_sql("SELECT * FROM Person_Person", export_conn_SDM)

for index, row in person_dimensie.iterrows():
    try:
        query = f"INSERT INTO Person_Person VALUES ({row['PersonID']}, '{row['PersonType']}', {row['NameStyle']}, '{row['Title'] if (row['Title'] != None) else None}', '{row['FirstName'].replace("'", "''") + (' ' + row['MiddleName'] if (row['MiddleName'] != 'None') else '') + ' ' + row['LastName'].replace("'", "''")}', '{row['Suffix']}', {row['EmailPromotion']})"
        export_cursor_result.execute(query)
    except pyodbc.Error:
        print(query)


export_conn1.commit()

In [866]:
Person_Address_dimensie = pd.read_sql("SELECT * FROM Person_Address", export_conn_SDM)

for index, row in Person_Address_dimensie.iterrows():
    try:
        query = f"INSERT INTO Person_Address VALUES ({row['AddressID']}, '{row['AddressLine1'].replace("'", "''")}', '{row['AddressLine2'].replace("'", "''") if (row['AddressLine2'] != None) else None}', '{row['City'].replace("'", "''")}', {row['StateProvinceID']}, '{row['PostalCode']}', {row['PersonID']})"
        export_cursor_result.execute(query)
    except pyodbc.Error:
        print(query)


export_conn1.commit()

In [867]:
Product_dimensie = pd.read_sql("SELECT * FROM Products", export_conn_SDM)
Category_dimensie = pd.read_sql("SELECT * FROM Categories", export_conn_SDM)
Suppliers_dimensie = pd.read_sql("SELECT * FROM Suppliers", export_conn_SDM)

Product_Category_dimensie =  pd.merge(Product_dimensie, Category_dimensie, left_on='CategoryID', how='outer', right_on='CategoryID')
Product_Category_Supplier_dimensie =  pd.merge(Product_Category_dimensie, Suppliers_dimensie, left_on='SupplierID', how='outer', right_on='SupplierID')

for index, row in Product_Category_Supplier_dimensie.iterrows():
    try:
        query = f"INSERT INTO Products VALUES ({row['ProductID']}, '{str(row['ProductName']).replace("'", "''")}', '{str(row['Description_x']).replace("'", "''") if(row['Description_x'] != None) else row['Description_x']}', {row['SupplierID'] if(row['SupplierID'] > 0 ) else 'Null'}, {row['CategoryID']if (row['CategoryID'] > 0) else 'Null'}, '{str(row['QuantityPerUnit']).replace("'", "''") if (row['QuantityPerUnit'] != None) else row['QuantityPerUnit']}', {row['UnitPrice'] if (row['UnitPrice'] != 'None' and row['UnitPrice'] > 0) else row['ListPrice'] if(row['ListPrice'] != 'None' and row['ListPrice'] > 0) else 'Null'}, {row['UnitsInStock'] if(row['UnitsInStock'] > 0 ) else 'Null'}, {row['UnitsOnOrder'] if(row['UnitsOnOrder'] > 0 ) else 'Null'}, {row['ReorderLevel'] if(row['ReorderLevel'] > 0 ) else 'Null'}, {0 if(row['Discontinued'] == 'False') else 1 if(row['Discontinued'] == 'True') else 'Null'}, '{row['color']}', '{row['ProductNumber']}', {row['MakeFlag'] if(row['MakeFlag'] > 0) else 'Null'}, {row['FinishedGoodsFlag'] if(row['FinishedGoodsFlag'] > 0) else 'Null'}, {row['SafetyStockLevel'] if(row['SafetyStockLevel'] > 0) else 'Null'}, {row['ReorderPoint'] if(row['ReorderPoint'] > 0) else 'Null'}, {row['StandardCost'] if(row['StandardCost'] > 0) else 'Null'}, '{row['SizeUnitMeasureCode']}', '{row['WeightUnitMeasureCode']}', {row['Weight']if (row['Weight'] > 0) else 'Null'}, {row['DaysToManufacture'] if(row['DaysToManufacture'] > 0) else 'Null'}, '{row['ProductLine']}', '{row['Class']}', '{row['prod_size']}', '{row['Style']}', '{row['SellStartDate']}', '{row['SellEndDate']}', '{row['DiscontinuedDate']}', {row['ProductModelID']if (row['ProductModelID'] > 0) else 'Null'}, '{row['CategoryName']}', '{str(row['Description_y']).replace("'", "''") if(row['Description_y'] != None) else row['Description_y']}', '{str(row['CompanyName']).replace("'", "''") if(row['CompanyName'] != None) else row['CompanyName']}', '{row['ContactName']}', '{row['ContactTitle']}', '{str(row['Address']).replace("'", "''") if(row['Address'] != None) else row['Address']}', '{row['City']}', '{row['Region']}', '{row['PostalCode']}', '{row['Country']}', '{row['Phone']}', '{row['Fax']}')"
        export_cursor_result.execute(query)
    except pyodbc.Error:
        print(query)


export_conn1.commit()

In [868]:
Production_BillOfMaterials_dimensie = pd.read_sql("SELECT * FROM Production_BillOfMaterials", export_conn_SDM)

for index, row in Production_BillOfMaterials_dimensie.iterrows():
    try:
        query = f"INSERT INTO Production_BillOfMaterials VALUES ({row['BillOfMaterialsID']}, {row['ProductAssemblyID'] if(row['ProductAssemblyID'] > 0) else 'Null'}, {row['ComponentID']}, '{row['StartDate']}', '{row['EndDate']}', '{row['UnitMeasureCode']}', {row['BOMLevel']}, {row['PerAssemblyQty']})"
        export_cursor_result.execute(query)
    except pyodbc.Error:
        print(query)


export_conn1.commit()

In [869]:
Sales_SalesTerritory_dimensie = pd.read_sql("SELECT * FROM Sales_SalesTerritory", export_conn_SDM)

for index, row in Sales_SalesTerritory_dimensie.iterrows():
    try:
        query = f"INSERT INTO Sales_SalesTerritory VALUES ({row['TerritoryID']}, '{row['Name'].replace("'", "''")}', '{row['CountryRegionCode']}', '{row['GroupName']}', {row['SalesYTD']}, {row['SalesLastYear']}, {row['CostYTD']}, {row['CostLastYear']})"
        export_cursor_result.execute(query)
    except pyodbc.Error:
        print(query)


export_conn1.commit()

In [870]:
Employee_dimensie = pd.read_sql("SELECT * FROM Employees", export_conn_SDM)
State_dimensie = pd.read_sql("SELECT * FROM State", export_conn_SDM)
Department_dimensie = pd.read_sql("SELECT * FROM Department", export_conn_SDM)

Employee_Department_dimensie =  pd.merge(Employee_dimensie, Department_dimensie, left_on='dept_id', how='outer', right_on='dept_id')
Employee_Department_State_dimensie =  pd.merge(Employee_Department_dimensie, State_dimensie, left_on='state_id', how='outer', right_on='state_id')

export_cursor_result.execute("ALTER TABLE Employees NOCHECK CONSTRAINT ALL")

for index, row in Employee_Department_State_dimensie.iterrows():
    try:
        if(row['EmployeeID'] > 0):
            query = f"INSERT INTO Employees VALUES ({row['EmployeeID']}, {row['PersonID'] if(row['PersonID'] > 0) else 'Null'}, '{row['NationalIDNumber']}', {row['manager_id'] if(row['manager_id'] != None and row['manager_id'] > 0) else row['ReportsTo'] if(row['ReportsTo'] != None and row['ReportsTo'] > 0) else 'Null'}, '{(row['emp_fname'] + ' ' +row['emp_lname']) if (row['emp_fname'] != None and row['emp_lname'] != None) else 'Null'}', '{row['Title']}', '{row['street']}', '{row['city']}', '{row['state_id'] if(row['state_id'] != None) else ''}', '{row['zip_code']}', '{row['Region_x'] if(row['Region_x'] != None) else row['Region_y'] if(row['Region_y'] != None) else None}', '{row['TitleCourtesy']}', '{row['phone']}', '{row['BirthDate']}', '{row['HireDate']}', '{row['Extension']}', '{row['Notes']}', {row['dept_id'] if(row['dept_id'] > 0) else 'Null'}, {row['ss_number'] if(row['ss_number'] != None and row['ss_number'] > 0) else 'Null'}, {row['salary'] if(row['salary'] != None and row['salary'] > 0) else 'Null'}, '{row['bene_health_ins']}', '{row['bene_life_ns']}', '{row['bene_day_care']}', '{row['sex']}', '{row['marital_status']}', '{row['status']}', '{row['LoginID']}', '{row['JobTitle']}', {row['OrganizationLevel'] if(row['OrganizationLevel'] > 0) else 'Null'}, {row['SalariedFlag'] if(row['SalariedFlag'] > 0) else 'Null'}, {row['VacationHours'] if(row['VacationHours'] > 0) else 'Null'}, {row['SickLeaveHours'] if(row['SickLeaveHours'] > 0) else 'Null'}, {row['CurrentFlag'] if(row['CurrentFlag'] > 0) else 'Null'}, '{row['Address']}', '{row['Country'] if(row['Country'] != None) else row['country'] if(row['country'] != None) else 'Null'}', {row['dept_head_id'] if(row['dept_head_id'] != None and row['dept_head_id'] > 0) else 'Null'}, '{row['dept_name']}', '{row['GroupName']}', '{row['state_name']}', '{row['state_capital']}')"
            export_cursor_result.execute(query)
    except pyodbc.Error:
        print(query)

export_cursor_result.execute("ALTER TABLE Employees WITH CHECK CHECK CONSTRAINT ALL")

export_conn1.commit()

In [ ]:
Customer_dimensie = pd.read_sql("SELECT * FROM Customers", export_conn_SDM)
State_dimensie = pd.read_sql("SELECT * FROM State", export_conn_SDM)
SalesStore_dimensie = pd.read_sql("SELECT * FROM Sales_Store", export_conn_SDM)

Customer_SalesStore_dimensie =  pd.merge(Customer_dimensie, SalesStore_dimensie, left_on='StoreID', how='outer', right_on='StoreID')
Customer_SalesStore_State_dimensie =  pd.merge(Customer_SalesStore_dimensie, State_dimensie, left_on='state_id', how='outer', right_on='state_id')

for index, row in Customer_SalesStore_State_dimensie.iterrows():
    try:
        if(str(row['CustomerID']) != 'nan'):
            query = f"INSERT INTO Customers VALUES ('{row['CustomerID']}', '{str(row['CompanyName']).replace("'", "''") if(row['CompanyName'] != None) else 'Null'}', '{row['ContactTitle']}', '{str(row['Address']).replace("'", "''")}', '{row['City']}', '{row['Region_x'] if(row['Region_x'] != None) else row['Region_y'] if(row['Region_y'] != None) else None}', '{row['PostalCode']}', '{row['Country'] if(row['Country'] != None) else row['country'] if(row['country'] != None) else 'Null'}', '{row['Phone']}', '{row['Fax']}', '{(str(row['fname']) + ' ' + str(row['lname']).replace("'", "''")) if (row['fname'] != None and row['lname'] != None) else 'Null'}', '{row['state_id'] if(row['state_id'] != None) else ''}', {row['PersonID'] if(row['PersonID'] > 0) else 'Null'}, {row['StoreID'] if(row['StoreID'] > 0) else 'Null'}, {row['TerritoryID'] if(row['TerritoryID'] != None and row['TerritoryID'] > 0) else 'Null'}, '{row['AccountNumber']}', '{row['state_name']}', '{row['state_capital']}', '{str(row['Name']).replace("'", "''")}', {row['SalesPersonID'] if(row['SalesPersonID'] != None and row['SalesPersonID'] > 0) else 'Null'})"
            export_cursor_result.execute(query)
    except pyodbc.Error:
        print(query)


export_conn1.commit()

,CustomerID,CompanyName,ContactTitle,Address,City,Region_x,PostalCode,Country,Phone,Fax,...,rowguid_x,ModifiedDate_x,Name,SalesPersonID,rowguid_y,ModifiedDate_y,state_name,state_capital,country,Region_y
0,50196,The Igloo,None,303 Roe Avenue,Edmonton,None,T5N 1S5,None,4035554884,None,...,None,None,NaN,NaN,NaN,NaN,Alberta,Edmonton,CAN,Canada
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Alaska,Juneau,USA,Western
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Alabama,Montgomery,USA,South
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Arkansas,Little Rock,USA,South
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Arizona,Phoenix,USA,Western
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20059,WARTH,Wartian Herkku,Accounting Manager,Torikatu 38,Oulu,None,90110,Finland,981-443655,981-443655,...,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20060,WELLI,Wellington Importadora,Sales Manager,"Rua do Mercado, 12",Resende,SP,08737-363,Brazil,(14) 555-8122,None,...,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20061,WHITC,White Clover Markets,Owner,305 - 14th Ave. S. Suite 3B,Seattle,WA,98128,USA,(206) 555-4112,(206) 555-4115,...,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20062,WILMK,Wilman Kala,Owner/Marketing Assistant,Keskuskatu 45,Helsinki,None,21240,Finland,90-224 8858,90-224 8858,...,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [872]:
Order_dimensie = pd.read_sql("SELECT * FROM Orders", export_conn_SDM)
Shippers_dimensie = pd.read_sql("SELECT * FROM Shippers", export_conn_SDM)

Order_Shippers_dimensie =  pd.merge(Order_dimensie, Shippers_dimensie, left_on='ShipVia', how='outer', right_on='ShipperID')

for index, row in Order_Shippers_dimensie.iterrows():
    try:
        query = f"INSERT INTO Orders VALUES ({row['OrderID']}, '{row['CustomerID']}', {row['EmployeeID']}, '{row['OrderDate']}', '{row['RequiredDate']}', '{row['ShippedDate']}', {row['ShipVia'] if(row['ShipVia'] != None and row['ShipVia'] > 0) else 'Null'}, {row['Freight'] if(row['Freight'] != None and row['Freight'] > 0) else 'Null'}, '{row['ShipName'].replace("'", "''") if(row['ShipName'] != None) else 'Null'}', '{row['ShipAddress'].replace("'", "''") if(row['ShipAddress'] != None) else 'Null'}', '{row['ShipCity']}', '{row['ShipRegion']}', '{row['ShipPostalCode']}', '{row['ShipCountry']}', '{row['Region']}', {row['RevisionNumber'] if(row['RevisionNumber'] != None and row['RevisionNumber'] > 0) else 'Null'}, {row['Status'] if(row['Status'] != None and row['Status'] > 0) else 'Null'}, {row['OnlineOrderFlag'] if(row['OnlineOrderFlag'] != None and row['OnlineOrderFlag'] > 0) else 'Null'}, '{row['SalesOrderNumber']}', '{row['PurchaseOrderNumber']}', '{row['AccountNumber']}', {row['TerritoryID'] if(row['TerritoryID'] != None and row['TerritoryID'] > 0) else 'Null'}, {row['BillToAddressID'] if(row['BillToAddressID'] != None and row['BillToAddressID'] > 0) else 'Null'}, {row['ShipToAddressID'] if(row['ShipToAddressID'] != None and row['ShipToAddressID'] > 0) else 'Null'}, {row['ShipMethodID'] if(row['ShipMethodID'] != None and row['ShipMethodID'] > 0) else 'Null'}, {row['CreditCardID'] if(row['CreditCardID'] > 0) else 'Null'}, '{row['CreditCardApprovalCode']}', {row['CurrencyRateID'] if(row['CurrencyRateID'] > 0) else 'Null'}, {row['SubTotal'] if(row['SubTotal'] != None and row['SubTotal'] > 0) else 'Null'}, {row['TaxAmt'] if(row['TaxAmt'] != None and row['TaxAmt'] > 0) else 'Null'}, {row['TotalDue'] if(row['TotalDue'] != None and row['TotalDue'] > 0) else 'Null'}, '{row['Comment']}', '{row['CompanyName']}', '{row['Phone']}')"
        export_cursor_result.execute(query)
    except pyodbc.Error:
        print(query)


export_conn1.commit()

In [873]:
Order_Details_dimensie = pd.read_sql("SELECT * FROM Order_Details", export_conn_SDM)

for index, row in Order_Details_dimensie.iterrows():
    try:
        query = f"INSERT INTO Order_Details VALUES ({row['SalesOrderDetailID']}, {row['OrderID']}, {row['ProductID']}, {row['UnitPrice'] if(row['UnitPrice'] != None and row['UnitPrice'] > 0) else 'Null'}, {row['OrderQty']}, {row['Discount'] if(row['Discount'] != None and row['Discount'] > 0) else 'Null'}, {row['LineTotal'] if(row['LineTotal'] != None and row['LineTotal'] > 0) else 'Null'}, {row['SpecialOfferID'] if(row['SpecialOfferID'] != None and row['SpecialOfferID'] > 0) else 'Null'}, '{row['ship_date']}')"
        export_cursor_result.execute(query)
    except pyodbc.Error:
        print(query)


export_conn1.commit()

In [874]:
CustomerCustomerDemo_dimensie = pd.read_sql("SELECT * FROM CustomerCustomerDemo", export_conn_SDM)
CustomerDemographics_dimensie = pd.read_sql("SELECT * FROM CustomerDemographics", export_conn_SDM)

CustomerCustomerDemo_CustomerDemographics_dimensie =  pd.merge(CustomerCustomerDemo_dimensie, CustomerDemographics_dimensie, left_on='CustomerTypeID', how='outer', right_on='CustomerTypeID')

for index, row in CustomerCustomerDemo_dimensie.iterrows():
    try:
        query = f"INSERT INTO CustomerCustomerDemo VALUES ('{row['CustomerID']}', {row['CustomerTypeID']},'{row['CustomerDesc'].replace("'", "''")}')"
        export_cursor_result.execute(query)
    except pyodbc.Error:
        print(query)


export_conn1.commit()

In [875]:
Bonus_dimensie = pd.read_sql("SELECT * FROM Bonus", export_conn_SDM)

for index, row in Bonus_dimensie.iterrows():
    try:
        query = f"INSERT INTO Bonus VALUES ({row['EmployeeID']}, '{row['bonus_date']}', {row['bonus_amount']})"
        export_cursor_result.execute(query)
    except pyodbc.Error:
        print(query)


export_conn1.commit()

In [876]:
EmployeeTerritories_dimensie = pd.read_sql("SELECT * FROM EmployeeTerritories", export_conn_SDM)
Territories_dimensie = pd.read_sql("SELECT * FROM Territories", export_conn_SDM)
Region_dimensie = pd.read_sql("SELECT * FROM Region", export_conn_SDM)

Territories_Region_dimensie =  pd.merge(Territories_dimensie, Region_dimensie, left_on='RegionID', how='outer', right_on='RegionID')
Territories_Region_EmployeeTerritories_dimensie =  pd.merge(EmployeeTerritories_dimensie, Territories_Region_dimensie, left_on='TerritoryID', how='outer', right_on='TerritoryID')

for index, row in Territories_Region_EmployeeTerritories_dimensie.iterrows():
    try:
        if(row['EmployeeID'] > 0):
            query = f"INSERT INTO EmployeeTerritories VALUES ({row['EmployeeID']}, {row['TerritoryID']}, '{row['TerritoryDescription']}', {row['RegionID']}, '{row['RegionDescription']}')"
            export_cursor_result.execute(query)
    except pyodbc.Error:
        print(query)


export_conn1.commit()

In [877]:
Purchasing_PurchaseOrderHeader_dimensie = pd.read_sql("SELECT * FROM Purchasing_PurchaseOrderHeader", export_conn_SDM)
Purchasing_Vendor_dimensie = pd.read_sql("SELECT * FROM Purchasing_Vendor", export_conn_SDM)

Purchasing_PurchaseOrderHeader_Purchasing_Vendor_dimensie = pd.merge(Purchasing_PurchaseOrderHeader_dimensie, Purchasing_Vendor_dimensie, left_on='VendorID', how='outer', right_on='VendorID')

for index, row in Purchasing_PurchaseOrderHeader_Purchasing_Vendor_dimensie.iterrows():
    try:
        if(row['PurchaseOrderID'] > 0):
            query = f"INSERT INTO Purchasing_PurchaseOrderHeader VALUES ({row['PurchaseOrderID']}, {row['RevisionNumber']}, {row['Status']}, {row['EmployeeID']}, {row['VendorID']}, {row['ShipMethodID']}, '{row['OrderDate']}', '{row['ShipDate']}', {row['SubTotal']}, {row['TaxAmt']}, {row['Freight']}, {row['TotalDue']}, '{row['AccountNumber']}', '{str(row['Name']).replace("'", "''")}', {row['CreditRating']}, '{row['PreferredVendorStatus']}', '{row['ActiveFlag']}')"
            export_cursor_result.execute(query)
    except pyodbc.Error:
        print(query)


export_conn1.commit()

In [878]:
Purchasing_PurchaseOrderDetail_dimensie = pd.read_sql("SELECT * FROM Purchasing_PurchaseOrderDetail", export_conn_SDM)

for index, row in Purchasing_PurchaseOrderDetail_dimensie.iterrows():
    try:
        query = f"INSERT INTO Purchasing_PurchaseOrderDetail VALUES ({row['PurchaseOrderDetailID']}, {row['PurchaseOrderID']}, '{row['DueDate']}', {row['OrderQty']}, {row['ProductID']}, {row['UnitPrice']}, {row['LineTotal']}, {row['ReceivedQty']}, {row['RejectedQty']}, {row['StockedQty']})"
        export_cursor_result.execute(query)
    except pyodbc.Error:
        print(query)


export_conn1.commit()

In [879]:
export_conn1.close()